## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip install --force-reinstall --no-cache-dir typing_extensions==4.11.0
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
## Sometimes needed in runpod to make sure it goes to the network volumne
import os

os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HOME"] = "/workspace/hf_home"

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import CLIPForImageClassification
from datasets import load_from_disk, concatenate_datasets
import pandas as pd
import numpy as np
import os
from collections import defaultdict
import copy
from typing import Optional
import gc

## Attribute Configurations

In [ ]:
# Custom Datasets Configuration
num_datasets = 8
num_classes = [47, 10, 43, 10, 45, 196, 397, 10]
vals = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
size_nums = [float('inf')]

# Regular for CLIP
domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
transformation = "Standard" # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
model_name = "CLIP_ViT_Vision"
folder = f"./Results/Multi_Class_Augmentation/{num_datasets}_{domain}/Entire_Transformation_Matrix_W"
os.makedirs(folder, exist_ok=True)
refer = CLIPForImageClassification.from_pretrained("openai/clip-vit-base-patch32")
indices = [i for i in range(refer.vision_model.encoder.config.num_hidden_layers)]
device = "cuda" if torch.cuda.is_available() else "cpu"

## Dataset Preparation

In [ ]:
def collate_fn(batch):
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images,
        "labels": labels
    }

In [ ]:
test_loader = {}
train = {}
full_train_size = {}
full_train_loader = {}

for i in range(num_datasets):
    train[i] = load_from_disk(f'/workspace/preprocessed/{dataset_name[i]}/train_processed')
    val = load_from_disk(f'/workspace/preprocessed/{dataset_name[i]}/val_processed')
    test = load_from_disk(f'/workspace/preprocessed/{dataset_name[i]}/test_processed')

    train[i].set_format(type='torch', columns=["image", "label", "pixel_values"])
    val.set_format(type='torch', columns=["image", "label", "pixel_values"])
    test.set_format(type='torch', columns=["image", "label", "pixel_values"])

    full_train = concatenate_datasets([train[i], val])
    full_train_size[i] = len(full_train)

    full_train_loader[i] = DataLoader(full_train, batch_size=32, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    test_loader[i] = DataLoader(test, batch_size=32, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

## Classes Preparation

In [ ]:
class Hooks(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.CLS = []
    
    def forward(self, images):
        self.CLS = []
        hidden_states = self.model.vision_model.embeddings(images)
        hidden_states = self.model.vision_model.pre_layrnorm(hidden_states)

        batch_size, seq_len, _ = hidden_states.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=hidden_states.device)
        attention_mask = attention_mask[:, None, None, :]
        casual_attention_mask = None

        for encoder_layer in self.model.vision_model.encoder.layers:
            layer_outputs = encoder_layer(hidden_states, attention_mask, casual_attention_mask, output_attentions=False)
            hidden_states = layer_outputs[0]
            self.CLS.append(hidden_states[:, 0, :])
        
        return self.CLS

In [ ]:
# https://github.com/huggingface/transformers/blob/main/src/transformers/models/clip/modeling_clip.py
class Augmented(torch.nn.Module):
    def __init__(self, model, classifier=None, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.classifier = classifier if classifier is not None else torch.nn.Linear(self.model.config.vision_config.hidden_size, num_classes)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage
    
    def forward(self, images):
        hidden_states = self.model.vision_model.embeddings(images)
        hidden_states = self.model.vision_model.pre_layrnorm(hidden_states)

        batch_size, seq_len, _ = hidden_states.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=hidden_states.device)
        attention_mask = attention_mask[:, None, None, :]
        casual_attention_mask = None

        for i, encoder_layer in enumerate(self.model.vision_model.encoder.layers):
            layer_outputs = encoder_layer(hidden_states, attention_mask, casual_attention_mask, output_attentions=False)
            hidden_states = layer_outputs[0]
            if i == self.transform_stage:
                if self.W is None:
                    self.W = torch.eye(hidden_states.shape[-1], device=hidden_states.device, dtype=hidden_states.dtype)
                cls = hidden_states[:, 0, :]
                cls = cls @ self.W
                hidden_states[:, 0, :] = cls
                break
        
        hidden_states = self.model.vision_model.post_layernorm(hidden_states[:, 0, :])
        logits = self.classifier(hidden_states)

        return logits, hidden_states # logits, post-layernorm cls

In [ ]:
def cosineSimilarity(fine_tuned_cls, aug_cls):
    eps = 1e-8
    out_aug = F.normalize(aug_cls, dim=1, eps=eps)
    out_fine = F.normalize(fine_tuned_cls, dim=1, eps=eps)

    cos_sim = (out_aug * out_fine).sum(dim=1).mean().item()
    return cos_sim

## Loading Models

In [ ]:
fine_tuned = {}
base = {}

fine_tuned_H = {}

for i in range(num_datasets):
    num_classes = vals[i]
    f_t = Augmented(copy.deepcopy(refer))
    f_t.load_state_dict(torch.load(f"./Fine_Tuned_Models/best_{model_name}_{dataset_name[i]}.pt"))

    base[i] = Augmented(copy.deepcopy(refer))
    base[i] = base[i].eval().to(device)

    fine_tuned_H[i] = Hooks(f_t.model).to(device)

    fine_tuned[i] = Augmented(copy.deepcopy(f_t.model), f_t.classifier)
    fine_tuned[i] = fine_tuned[i].eval().to(device)

base_H = Hooks(copy.deepcopy(refer)).to(device)

## Dataset Prep

In [ ]:
filtered_train = {}

for i in range(num_datasets):
    labels = train[i]["label"]

    label_to_indices = defaultdict(list)

    for idx, label in enumerate(labels):
        label = int(label)
        label_to_indices[label].append(idx)
    
    filtered_train[i] = {
        label: train[i].select(indices) for label, indices in label_to_indices.items()
    }

In [ ]:
def extract_vectors(train_loader, base_H, fine_tuned_H):
    Z0 = {i: [] for i in indices}
    Z1 = []

    for k in range(num_datasets):
        with torch.no_grad():
            for batch in tqdm(train_loader[k], desc="Extracting"):
                images = batch["pixel_values"].to(device, non_blocking=True)

                if domain == "Fine_Tuned_Layer_Skipping":
                    out_fine_tuned = fine_tuned_H[k](images)
                    
                    for i in indices:
                        Z0[i].append(out_fine_tuned[i].float().cpu())
                    Z1.append(out_fine_tuned[-1].float().cpu())
                else:
                    out_base = base_H(images)
                    out_fine_tuned = fine_tuned_H[k](images)

                    for i in indices:
                        Z0[i].append(out_base[i].float().cpu())
                    Z1.append(out_fine_tuned[-1].float().cpu())
        del fine_tuned_H[k]
        torch.cuda.empty_cache()
        del train[k]
        gc.collect()
        del train_loader[k]
        gc.collect()
    
    del base_H
    del fine_tuned_H
    del train
    del train_loader
    torch.cuda.empty_cache()
    gc.collect()
    
    Z1_last = torch.cat(Z1)
    Z1 = Z1_last.cpu().numpy()

    W = {}
    resid = {}

    for i, val in Z0.items():
        val = torch.cat(val)
        val = val.cpu().numpy()
        W[i], resid[i], _, _ = np.linalg.lstsq(val, Z1, rcond=None)
    
    return W, resid

In [ ]:
def augment_models(transform_type, W=None):
    augmented = []
    if W is None:
        W = {i: None for i in indices}

    for k in range(num_datasets):
        aug = {}
        for i in indices:
            reference = copy.deepcopy(refer)
            if domain == "Fine_Tuned_Layer_Skipping":
                reference = copy.deepcopy(fine_tuned[k].model)
            if transform_type == "Standard":
                model = Augmented(copy.deepcopy(reference), classifier=fine_tuned[k].classifier, W=W[i], transform_stage=i)
            elif transform_type == "Base_Fine_Tuned_Classifier":
                model = Augmented(copy.deepcopy(reference), classifier=fine_tuned[k].classifier, transform_stage=i)
            model = model.eval().to(device)
            aug[i] = model
        augmented.append(aug)
    return augmented

In [ ]:
def evaluate(augmented, test_loader, fine_tuned, base):
    base_acc = {}
    fine_tuned_acc = {}
    correct = {}
    co_sim_cls = {}

    for k in range(num_datasets):
        aug = augmented[k]
        correct_base = 0
        correct_fine_tuned = 0
        total_samples = 0

        correct[k] = {i: 0 for i in indices}
        co_sim_cls[k] = {i: [] for i in indices}

        for batch in tqdm(test_loader[k], desc="Evaluating"):
            images = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            total_samples += labels.size(0)

            # Base Model
            logits_base, cls_base = base[k](images)
            predicted = logits_base.argmax(dim=1)
            correct_base += (predicted == labels).sum().item()

            # Fine-Tuned Model
            logits_fine_tuned, cls_fine_tuned = fine_tuned[k](images)
            predicted = logits_fine_tuned.argmax(dim=1)
            correct_fine_tuned += (predicted == labels).sum().item()

            # Augmented Models
            for i in indices:
                logits_aug, cls_aug = aug[i](images)
                predicted = logits_aug.argmax(dim=1)
                correct[k][i] += (predicted == labels).sum().item()
                co_sim_cls[k][i].append(cosineSimilarity(cls_fine_tuned, cls_aug))

        base_acc[k] = correct_base / total_samples
        fine_tuned_acc[k] = correct_fine_tuned / total_samples

        for i in indices:
            correct[k][i] = correct[k][i] / total_samples
            co_sim_cls[k][i] = np.mean(co_sim_cls[k][i])

        print(f"Augmented {model_name[k]} on {dataset_name[k]} Results")
        for i in indices:
            print(f"\tAugmented {i} - Last ({indices[-1]}) Layer Accuracy: {correct[k][i]}")
            print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i} Layer: {co_sim_cls[k][i]:.4f}")
        print(f"Base Accuracy: {base_acc[k]:.4f}")
        print(f"Fine-Tuned Accuracy: {fine_tuned_acc[k]:.4f}")

        del aug
        del augmented[k]
        del test_loader[k]
        del base[k]
        del fine_tuned[k]
        torch.cuda.empty_cache()
        gc.collect()
    
    del augmented
    del test_loader
    del base
    del fine_tuned
    torch.cuda.empty_cache()
    gc.collect()

    return base_acc, fine_tuned_acc, correct, co_sim_cls

In [ ]:
def save_results(base_acc, fine_tuned_acc, correct, co_sim_cls, train_size, save_W=False, W=None):
    for k in range(num_datasets):
        name = f"{dataset_name[k]}_{transformation}_Results.json"
        
        data = {
            'Classification_Accuracy': [correct[k][i] for i in indices],
            'CLS_Cosine_Similarity': [co_sim_cls[k][i] for i in indices],
            'Base_Accuracy': [base_acc[k]] * len(indices),
            'Fine_Tuned_Accuracy': [fine_tuned_acc[k]] * len(indices),
            'Train_Size': [train_size[k]] * len(indices),
        }

        df = pd.DataFrame(data, index=indices)
        path = os.path.join(folder, name)
        df.to_json(path, orient="records", indent=2)

    if save_W:
        name = "Task_Matrix_W.json"
        data = {
            'W': [W[i] for i in indices]
        }

        df = pd.DataFrame(data, index=indices)
        path = os.path.join(folder, name)
        df.to_json(path, orient="records", indent=2)

## Evaluating

In [ ]:
train_loader = {}
train_size = {}

for s in size_nums:
    image_per_label = s

    sorted = {i: [] for i in range(num_datasets)}
    for k in range(num_datasets):
        sort = []
        for i in range(vals[k]):
            num_indices = min(image_per_label, len(filtered_train[k][i]))
            sort.append(filtered_train[k][i].select(range(num_indices)))
        train_dataset = concatenate_datasets(sort)
        train_size[k] = len(train_dataset)
        train_loader[k] = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    
    models = f"{dataset_name[0]}"
    total_size = f"{train_size[0]}"
    for i in range(1, num_datasets):
        models += f"; {dataset_name[i]}"
        total_size += f"; {train_size[i]}"

    print(f"Results for {model_name} on {models}: {total_size} Training Images ({image_per_label} images/label)")
    # W, resid = extract_vectors(train_loader, base_H, fine_tuned_H)
    W = {}
    W_file = pd.read_json(f"{folder}/Task_Matrix_W.json")
    for i in indices:
        W[i] = np.array(W_file["W"][i])
    
    aug = augment_models(transformation, W=W)
    base_acc, fine_tuned_acc, correct, co_sim_cls = evaluate(aug, test_loader, fine_tuned, base)
    save_results(base_acc, fine_tuned_acc, correct, co_sim_cls, train_size, save_W=False)